# Data Warehousing Project 2: ETF Recommender System w GenAI

## Reference Resources
- **MorningStar Fund Similarity Screener (Presentation from industry)**
  * Video: https://drive.google.com/file/d/1XqJ9QworB28evacdTa8FeYP_cmxzcbc0/view?usp=sharing
- **"Personalized Machine Learning"**
  * https://cseweb.ucsd.edu/~jmcauley/pml/pml_book.pdf
- **Walkthrough video for first part (initially project 2)**
  * https://drive.google.com/file/d/1Qw2DGrWU-Z-34YUiyEBt1ce0kmR3MSb-/view?usp=sharing
- **Walkthrough video for second part (initially project 3)**
  * https://drive.google.com/file/d/1qp61UB0dJCvx03ArxUHExJ3kfVvrshws/view?usp=sharing

---

## Business Context

It's your second week at **Charles Schwab's Intelligent Portfolios** division, and the group chat is already on fire. Schwab just lost a major institutional client — a \$400M pension fund — to a competitor whose platform could answer a deceptively simple question: *"We hold SPY. What else is out there that's cheaper and does the same thing?"*

Your Managing Director calls an all-hands: *"Vanguard and BlackRock have spent millions building recommendation engines that keep clients in their ecosystem. Every time a client searches for an ETF on someone else's platform, we're one click away from losing their assets. We need our own recommender — and we need it yesterday."*

Here's the problem: the ETF market has exploded. There are now over **3,000 ETFs** in the U.S. alone, with overlapping holdings, confusing names, and expense ratios that range from 0.03% to over 1.00%. A client holding **\$10,000,000** in a high-fee ETF at 0.75% is paying **\$75,000 per year** in fees. If you can identify a nearly identical ETF at 0.10%, you just saved that client **\$65,000 annually** — without changing their market exposure by a single basis point.

The Head of Product pulls your team aside: *"Build me a proof of concept. I want to type in any ETF ticker and instantly see the most similar alternatives ranked by holdings overlap, with fee savings highlighted. If this works, we're pitching it to the C-suite next quarter as a client retention tool. Your data pipeline needs to be production-grade — we can't recommend stale holdings or crash on bad data. And one more thing: I want the system to explain* why *two ETFs are similar, in plain English. Our advisors aren't quants — they need to articulate this to clients over the phone."*

That last requirement — generating natural-language explanations of ETF similarity — is where **generative AI** enters your pipeline. You'll use an LLM to produce strategy summaries for each ETF based on its holdings, turning raw ticker lists into advisor-ready descriptions.

This is your chance to build something real. Ship it.

---

## Learning Objectives
- Master web scraping with iterative problem-solving (starting simple, handling failures)
- Design efficient NoSQL database schemas for query performance
- Implement robust exception handling for real-world data collection
- Build a practical recommendation system using similarity metrics
- Use generative AI (LLMs) to produce structured financial summaries from raw data
- Implement self-healing data pipelines that respect API rate limits

---

## Database Structure
We will create two MongoDB collections with carefully designed document structures:
1. **etf_master** — A comprehensive list of all ETFs including their metadata
2. **etf_holdings** — For each Equity ETF, a document containing all of its holdings and an AI-generated strategy summary

---

## Part 1: ETF Master Collection — Building the Universe of ETFs

### Objective
Create a comprehensive collection of ALL ETFs with their basic metadata. The most critical field is **AUM (assets under management)** — you'll use this to filter for large Equity ETFs. Other fields (price, volume, asset class, expense ratio) depend on your source. Store all ETFs now, filter for Equity ETFs later when building holdings.

### Data Source Priority

The goal is to find Equity (or Equities) ETFs that have an AUM greater than \$2bn.

1. **Primary**: Finviz.com — Reliable with pagination support
   - General URL pattern: `https://finviz.com/screener.ashx?v=111&f=ind_exchangetradedfund&r={page}`
   - Pages start at 1, 21, 41, etc. (increments of 20)
   - This page gets you ETF categories: `https://finviz.com/screener.ashx?v=181&f=ind_exchangetradedfund`
   - This page gets you ETF AUM: `https://finviz.com/screener.ashx?v=191&f=ind_exchangetradedfund`
   - You can sort any of the pages above by AUM if needed: `https://finviz.com/screener.ashx?v=181&f=ind_exchangetradedfund&o=-e.assetsundermanagement`

2. **Alternative Source**: https://stockanalysis.com/etf/ with requests + BS4
   - Same goal of getting Equity ETFs that have an AUM greater than \$2bn.
   - ETF data is embedded in the page HTML — explore the page source to find it
   - Selenium is NOT required for this source
   - Fields available include: Symbol, Fund Name, Asset Class, Assets (AUM), and more

3. **Fallback (0 points)**: Bloomberg Excel file
   - https://www.dropbox.com/s/1a4u95oj30x68k8/ETF1.xlsx?raw=1
   - Only use this to continue with other parts if stuck

### Implementation Recommendations

#### 1. Progressive Web Scraping Approach
Start simple and build complexity:
```python
# Step 1: Try pd.read_html() first
# Step 2: If that fails, use requests + BeautifulSoup
# Step 3: Add user-agent rotation if getting 403 errors
# Step 4: Implement pagination
# Step 5: Add caching to avoid repeated requests
```

#### 2. Recommended Setup
```python
# Install packages
pip install requests beautifulsoup4 pandas pymongo random-user-agent

# User agent rotation to avoid blocks
from random_user_agent.user_agent import UserAgent
from random_user_agent.params import SoftwareName, OperatingSystem

software_names = [SoftwareName.CHROME.value]
operating_systems = [OperatingSystem.WINDOWS.value, OperatingSystem.LINUX.value]
user_agent_rotator = UserAgent(software_names=software_names, operating_systems=operating_systems, limit=100)
```

#### 3. Exception Handling (REQUIRED)
Your code must handle these common issues:
- **Network errors**: 403 Forbidden, 429 Too Many Requests, timeouts
- **Parsing errors**: Missing tables, different HTML structures
- **Data inconsistencies**: Missing columns, unexpected formats

Example structure:
```python
def getETFMaster(page, retry=10):
    try:
        # Request logic here
        response = requests.get(url, headers={'User-Agent': user_agent_rotator.get_random_user_agent()})
        time.sleep(0.1)  # Prevent rate limiting

        if response.status_code == 200:
            # Parse and return data
            pass
        elif response.status_code == 429:
            # Rate limited - wait longer
            time.sleep(2 ** (10 - retry))
            return getETFMaster(page, retry-1)

    except Exception as e:
        if retry > 0:
            return getETFMaster(page, retry-1)
        return None
```

#### 4. Data Cleaning Recommendations
Web data is messy! Recommended approaches:
- **Standardize column names (CRITICAL for MongoDB)**:
  - All keys must be single-word snake_case — `'Asset Class'` → `'asset_class'`
  - Remove special characters (., /, \$) as they cause MongoDB issues
  - Remove spaces (they complicate indexing and querying)
  - Be consistent: decide on snake_case and use it everywhere
- **Parse AUM strings to numbers**:
  - `'$2.15B'` → `2150000000.0`, `'$517.3M'` → `517300000.0`
  - This is essential — you cannot do `{'aum': {'$gte': 2e9}}` if AUM is a string
- **Convert all data types**:
  - Remove \$, commas, % from numeric fields
  - Convert strings to float/int where appropriate
  - Replace '-' with NaN for missing values
- **Remove duplicate ETF tickers**: Check for and handle duplicate entries
- **Drop useless columns**: Remove columns with no variation

#### 5. Add Query Date to Documents

**IMPORTANT**: Add a date string to track when data was collected (one load per day). This lets you track how ETF AUM evolves over time — a key business metric.
```python
from datetime import datetime

# Use date string format for daily granularity
today_str = datetime.now().strftime('%Y%m%d')  # e.g., '20260315'
```

#### 6. MongoDB Document Design Principles

Different sources provide different fields. Finviz gives you price, volume, and change. StockAnalysis gives you Asset Class and AUM. Regardless of source, your documents must follow these rules:

**Key normalization requirements:**
- All keys must be **single-word with no spaces or special characters** — pick either `snake_case` or `camelCase` and be consistent across your entire project
  - ❌ `'Asset Class'`, `'Fund Name'`, `'%Change'`
  - ✅ `'asset_class'`, `'fund_name'`, `'pct_change'` (snake_case)
  - ✅ `'assetClass'`, `'fundName'`, `'pctChange'` (camelCase)
- MongoDB cannot index or query keys with dots (`.`) or dollar signs (`$`) — clean these out
- Numeric fields must be stored as **numbers, not strings** — `2150000000.0` not `'$2.15B'`
- Every document must include `etf_ticker` and `query_date`

**The two most important fields are `query_date` and `aum`**. In finance, data without a date is useless — you need to know *when* something was true, not just *what* was true. The `query_date` lets you track how ETF AUM evolves over time, which is critical for any time-series analysis. The `aum` field is what you'll use to filter for large Equity ETFs. Make sure to parse strings like `'$2.15B'` and `'$517.3M'` into numeric values so MongoDB can perform range queries and sorts.

**Example document (fields will vary by source):**
```python
{
    'etf_ticker': 'SPY',
    'fund_name': 'SPDR S&P 500 ETF Trust',
    'asset_class': 'Equity',
    'aum': 502000000000.0,       # Parsed from '$502B' — must be numeric!
    'expense_ratio': 0.000945,    # 0.0945% = 9.45 basis points — be careful with the conversion!
    'query_date': '20260315'     # YYYYMMDD format
    # ... include whatever other fields your source provides
}
```

**Create proper indexes:**
```python
# Compound unique index - ensures only one entry per ETF per day
etf_master_col.create_index([('query_date', -1), ('etf_ticker', 1)], unique=True)
# Additional indexes for common queries
etf_master_col.create_index('etf_ticker')
etf_master_col.create_index('aum')
```

#### 7. Index Performance Analysis (REQUIRED)

This is a data warehousing class — you must demonstrate that you understand **why** indexes matter, not just how to create them.

**Required demonstration using `.explain()`:**

Run the same query **before and after** creating an index, and compare the query plans:

```python
# Step 1: Run a query WITHOUT an index and capture the explain plan
explain_no_index = etf_master_col.find(
    {'aum': {'$gte': 2e9}}
).explain()
print("Without index:", explain_no_index['executionStats']['executionTimeMillis'], 'ms')
print("Documents examined:", explain_no_index['executionStats']['totalDocsExamined'])

# Step 2: Create the index
etf_master_col.create_index('aum')

# Step 3: Run the SAME query and capture the explain plan again
explain_with_index = etf_master_col.find(
    {'aum': {'$gte': 2e9}}
).explain()
print("With index:", explain_with_index['executionStats']['executionTimeMillis'], 'ms')
print("Documents examined:", explain_with_index['executionStats']['totalDocsExamined'])
```

**In your presentation, discuss:**
- **Without index**: MongoDB performs a **COLLSCAN** (collection scan) — it reads every single document to check if `aum >= 2e9`. If you have 3,000 ETFs × 30 days = 90,000 documents, it examines all 90,000.
- **With index**: MongoDB performs an **IXSCAN** (index scan) — it jumps directly to the relevant range. It might examine only 200 documents.
- **The tradeoff**: Indexes speed up reads but slow down writes (every `insert` or `update` must also update the index) and consume additional storage. For a data warehouse where you load once and query many times, this is almost always worth it.
- Show the `winningPlan` from both explain outputs to illustrate COLLSCAN vs IXSCAN.

### Common Pitfalls to Avoid
- Finviz tables have class `screener_table` — make sure to find the right one
- Not all 23 tables on the page are useful — identify the correct one
- Handle pagination properly (remember it's 1, 21, 41, not 1, 2, 3)
- Cache responses to avoid hitting the server repeatedly
- If using Google Colab and get 429 errors, "disconnect and delete runtime" gives new IP

---

## Part 2: ETF Holdings Collection — Web Mining

### Objective
Retrieve holdings data for Equity ETFs only. Skip ETFs that primarily hold bonds, futures, swaps, or other non-equity assets.

### Data Source Priority
Always prefer primary sources (ETF providers) over secondary sources:

1. **iShares** (Best source — direct from provider, requires Selenium)
   - Use Selenium to navigate the iShares product screener and extract ETF URLs
   - Filter for Equity funds on their website
   - Two-step process: (1) Scrape the ETF list page to get each fund's sub-URL, (2) construct the CSV download URL from the sub-URL
   - The CSV download URL follows a pattern — once you have a fund's page URL, the holdings CSV is accessible at a predictable path
   - This was covered in class; refer to the lecture notebook for guidance

2. **Invesco** (Requires Selenium)
   - Solution provided here — study the approach and adapt it:
   - https://drive.google.com/file/d/1VgXEXFZP_0IWAdy8ykSvuH33AF8ff2xO/view?usp=sharing

3. **SSGA (State Street Global Advisors)**
   ```python
   ! wget https://www.ssga.com/us/en/intermediary/etfs/library-content/products/fund-data/etfs/us/us_spdrallholdings.zip
   ! unzip us_spdrallholdings.zip
   # Process Excel files for each SPDR ETF
   ```
   **Historical SSGA data**: Professor Low has been collecting these zip files over time. Two snapshots are provided (August 2025 and March 2026). Load both into your `etf_holdings` MongoDB collection with their respective `query_date` values. This is a core data warehousing concept — your `query_date` field makes it possible to track changes over time.

  * Aug 2025: https://drive.google.com/file/d/1VDhmL_IxXdCBpF6MVcuBLc7Gla8NrzEA/view?usp=sharing
  * Mar 2026: https://drive.google.com/file/d/1cgzIfQQP1R1UfKFwmw761whCc1ejj_JM/view?usp=sharing

   **Required temporal analysis — XLK as a case study**: The file `holdings-daily-us-en-xlk.xlsx` (Technology Select Sector SPDR) tells a compelling story about rotation within tech. Your task:

   1. **Load both snapshots into MongoDB** with different `query_date` values
   2. **Use MongoDB queries** to retrieve XLK holdings from each date and compute which stocks gained or lost the most weight between snapshots
   3. **Identify your top 3 gainers and top 3 losers** by weight change
   4. **Feed the results to your LLM** using the prompt template below

   ```python
   # Fill in YOUR top 3 gainers and losers from your MongoDB query results
   top_3_gainers = "STOCK1 (+X.XX), STOCK2 (+X.XX), STOCK3 (+X.XX)"   # YOUR RESULTS
   top_3_losers  = "STOCK4 (-X.XX), STOCK5 (-X.XX), STOCK6 (-X.XX)"   # YOUR RESULTS

   prompt = f"""You are a senior equity analyst at a wealth management firm.
   A client holds XLK (Technology Select Sector SPDR ETF) and noticed their
   quarterly statement looks different. They want to understand what changed.

   Between August 2025 and March 2026, the biggest weight increases were:
   {top_3_gainers}

   The biggest weight decreases were:
   {top_3_losers}

   In 3-4 sentences that a non-technical client could understand:
   1. What sub-industries within tech do the gainers vs losers belong to
      (e.g., cloud software, semiconductor equipment, memory chips, AI processors)
   2. What is driving this rotation within the tech sector
   3. Whether this is a fundamental shift or normal index rebalancing

   Be concise. Do not use jargon without defining it."""

   response = model.generate_content(prompt)
   ```

   **In your presentation, concisely discuss:**
   - Your top 3 gainers and top 3 losers from your MongoDB query
   - Whether the shift represents a sub-industry rotation within tech (e.g., cloud/software vs semiconductors/equipment)
   - Whether the LLM's explanation was accurate or hallucinated — and how you verified it

   This analysis demonstrates why data warehousing matters — without `query_date`, you'd only see today's snapshot and miss the story entirely.

4. **StockAnalysis.com** (Last resort — secondary source)
   - Example: `https://stockanalysis.com/etf/spy/holdings/`
   - Limited to 500 holdings
   - Less accurate/potentially stale data

### MongoDB Document Design for Holdings

**❌ BAD Design — Don't do this:**
```python
# Stock ticker as value - hard to index and query
{'IYY': {'AAPL': 5.2, 'MSFT': 4.8}}
```

**✅ GOOD Design:**
```python
{
    'etf_ticker': 'IYY',
    'holdings': [
        {'stock_ticker': 'AAPL', 'weight': 5.2},
        {'stock_ticker': 'MSFT', 'weight': 4.8},
        {'stock_ticker': 'GOOGL', 'weight': 3.1}
    ],
    'query_date': '20260315',
    'source': 'ishares',
    'strategy_summary': None  # To be populated by LLM in Part 4
}
```

**Create compound indexes following ESR principle:**
```python
# Compound unique index - one holdings document per ETF per day
etf_holdings_col.create_index([('query_date', -1), ('etf_ticker', 1)], unique=True)

# Indexes for common queries
etf_holdings_col.create_index('etf_ticker')
etf_holdings_col.create_index('holdings.stock_ticker')
```

### Data Collection Strategy
**IMPORTANT**: Check if today's data already exists before fetching!

```python
from datetime import datetime

today_str = datetime.now().strftime('%Y%m%d')

# Check what ETFs already have today's data
existing_etfs = set(
    etf_holdings_col.distinct('etf_ticker', {'query_date': today_str})
)

# Only query for ETFs missing today's data
for etf in all_equity_etfs:
    if etf not in existing_etfs:
        holdings_data = fetch_holdings(etf)

        doc = {
            'etf_ticker': etf,
            'holdings': holdings_data,
            'query_date': today_str,
            'source': 'ishares',
            'strategy_summary': None
        }

        etf_holdings_col.update_one(
            {'etf_ticker': etf, 'query_date': today_str},
            {'$set': doc},
            upsert=True
        )

        time.sleep(0.5)
```

### Data Cleaning for Holdings
Different sources have different formats. Ensure consistency:
- **Column names**: Map variations to consistent names
  - 'Ticker'/'Symbol' → 'stock_ticker' (distinguish from etf_ticker!)
  - 'Weight (%)'/'Weighting' → 'weight'
- **Data types**:
  - Ensure weights are floats, not strings (enables numeric queries in MongoDB)
  - Remove % signs and convert to decimal (5.2% → 5.2 or 0.052)
- **Stock ticker validation**:
  - US stock tickers are typically ≤4 letters, all caps, alphabetic only
  - Remove invalid entries like 'n/a', spaces, special characters
  - Example: ' AAPL ' → 'AAPL'
- **Weight normalization**: Ensure weights sum to approximately 100% (or 1.0)

### Handling Rate Limits
```python
# Rotate user agents
headers = {'User-Agent': user_agent_rotator.get_random_user_agent()}

# Add delays between requests
time.sleep(0.5)

# If getting 429 errors on Google Colab:
# 1. Go slower (increase sleep time)
# 2. "Disconnect and delete runtime" for new IP
# 3. Switch to different data source temporarily
```

---

## Part 3: MongoDB Best Practices

### Connection Setup
```python
from pymongo import MongoClient
import os

# Don't hardcode credentials!
URI = os.getenv('MONGODB_URI')  # Set in environment
client = MongoClient(URI)
db = client['etf_project']
etf_master_col = db['etf_master']
etf_holdings_col = db['etf_holdings']
```

### Bulk Operations vs Individual Inserts
```python
# ❌ Slow - individual inserts
for doc in documents:
    collection.insert_one(doc)

# ✅ Fast - bulk insert
collection.insert_many(documents)

# ✅ Best - upsert to handle duplicates with date
from datetime import datetime

today_str = datetime.now().strftime('%Y%m%d')

for doc in documents:
    doc['query_date'] = today_str
    collection.update_one(
        {'etf_ticker': doc['etf_ticker'], 'query_date': today_str},
        {'$set': doc},
        upsert=True
    )
```

### Benefits of this approach:
1. **Prevents duplicate daily loads**: The unique index on (query_date, etf_ticker) ensures only one entry per ETF per day
2. **Historical tracking**: Can maintain price history over multiple days
3. **Efficient resumption**: Easy to check which ETFs already have today's data
4. **Clear data freshness**: String date format makes it obvious when data was collected

---

## Part 4: AI-Generated Strategy Summaries (REQUIRED — New for Spring 2026)

### Objective
For each ETF in your holdings collection, generate a plain-English summary of the fund's investment strategy using an LLM. This transforms raw ticker lists into advisor-ready descriptions that explain *what* the ETF does and *why* someone might hold it.

### Why This Matters
In practice, a recommender system that says "QQQ is 83% similar to XLK" is only half the story. An advisor needs to tell a client: *"Both funds are heavily concentrated in mega-cap technology — Apple, Microsoft, and Nvidia make up over 30% of each. The key difference is QQQ includes non-tech Nasdaq names like Costco and PepsiCo, while XLK is pure tech sector."* Generating this kind of narrative from structured holdings data is exactly the kind of task where LLMs excel.

### LLM Setup

You can use **either** of these free options. Document which one you used per the course AI disclosure policy.

**Option A: Google Gemini (Free API — recommended for Colab)**

> **Note:** Gemini 2.0 Flash was retired in March 2026. Use `gemini-2.5-flash-lite` for the highest free-tier quota (up to 1,000 requests/day).

```python
import google.generativeai as genai

# Get your free API key at https://aistudio.google.com/apikey
# Do NOT commit your key to GitHub or share it in your notebook.
# Use environment variables or Colab Secrets.
genai.configure(api_key='YOUR_API_KEY_HERE')
model = genai.GenerativeModel('gemini-2.5-flash-lite')
```

**Option B: Ollama with Gemma 3n (Free — runs locally, no rate limits)**

If you prefer to run a model on your own machine (no API key needed, no rate limits):
```bash
# Install from https://ollama.com/download
ollama pull gemma3n:e4b    # 7.5GB download, effective 4B parameters — recommended
# or for lower-spec machines:
ollama pull gemma3n:e2b    # 5.6GB download, effective 2B parameters
```
Gemma 3n uses selective parameter activation, meaning the model is larger on disk but only activates 2B or 4B parameters per inference — so it runs efficiently on laptops without a GPU.

```python
import requests

def query_ollama(prompt, model="gemma3n:e4b"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    return response.json()["response"]
```

**Other LLMs**: You are free to use any LLM you have access to — OpenAI, Claude (Anthropic), Mistral, etc. We cover Gemini and Ollama in class because they are free, but grading is based on your pipeline design and output quality, not which model you chose. Just document your choice in your AI disclosure.

### Prompt Template

```python
def build_summary_prompt(etf_ticker, holdings):
    """Build a prompt for the LLM to summarize an ETF's strategy."""
    # Format top holdings as a readable list
    top_holdings = holdings[:15]  # Send top 15 to keep prompt short
    holdings_str = "\n".join(
        f"  {h['stock_ticker']}: {h['weight']:.1f}%"
        for h in top_holdings
    )

    prompt = f"""You are a financial analyst writing for investment advisors.
Given the top holdings of ETF '{etf_ticker}', write a 2-3 sentence summary
of the fund's likely investment strategy. Focus on:
- What sectors or themes the holdings represent
- Whether the fund appears concentrated or diversified
- The general market cap tilt (large-cap, mid-cap, small-cap)

Top holdings:
{holdings_str}

Write only the summary, no preamble."""

    return prompt
```

### Self-Healing Pipeline (REQUIRED)

Your pipeline must handle LLM rate limits gracefully. The pattern: load all holdings first, then cure missing summaries in a separate pass. If you hit a rate limit, stop and resume later — the pipeline picks up where it left off.

```python
import time

def generate_missing_summaries(etf_holdings_col, model, today_str, max_calls=50):
    """
    Self-healing summary generator.
    Finds ETFs missing strategy_summary and fills them in,
    respecting rate limits. Run repeatedly until all are filled.
    """

    # Find documents that have holdings but no summary yet
    missing = etf_holdings_col.find({
        'query_date': today_str,
        '$or': [
            {'strategy_summary': None},
            {'strategy_summary': {'$exists': False}}
        ]
    })

    filled = 0
    for doc in missing:
        if filled >= max_calls:
            print(f"Reached {max_calls} calls. Run again later to continue.")
            break

        etf_ticker = doc['etf_ticker']
        holdings = doc['holdings']

        try:
            prompt = build_summary_prompt(etf_ticker, holdings)
            response = model.generate_content(prompt)
            summary = response.text.strip()

            # Update the document with the summary
            etf_holdings_col.update_one(
                {'etf_ticker': etf_ticker, 'query_date': today_str},
                {'$set': {'strategy_summary': summary}}
            )

            filled += 1
            print(f"[{filled}] {etf_ticker}: {summary[:80]}...")

            # Respect rate limits: ~5-10 requests per minute for free tier
            time.sleep(8)

        except Exception as e:
            if '429' in str(e) or 'quota' in str(e).lower():
                print(f"Rate limited at {etf_ticker}. Stopping. Run again later.")
                break
            else:
                print(f"Error on {etf_ticker}: {e}. Skipping.")
                continue

    # Report progress
    total = etf_holdings_col.count_documents({'query_date': today_str})
    done = etf_holdings_col.count_documents({
        'query_date': today_str,
        'strategy_summary': {'$ne': None, '$exists': True}
    })
    print(f"\nProgress: {done}/{total} ETFs have summaries.")
```

### How to Use It

```python
# Run this cell. If it stops due to rate limits, just run it again.
# It will pick up where it left off — that's the self-healing part.
generate_missing_summaries(etf_holdings_col, model, today_str, max_calls=50)
```

### What Your MongoDB Document Should Look Like After Part 4

```python
{
    'etf_ticker': 'QQQ',
    'holdings': [
        {'stock_ticker': 'AAPL', 'weight': 8.9},
        {'stock_ticker': 'MSFT', 'weight': 8.1},
        {'stock_ticker': 'NVDA', 'weight': 7.6},
        # ... more holdings
    ],
    'query_date': '20260315',
    'source': 'ishares',
    'strategy_summary': 'QQQ is a large-cap growth fund heavily concentrated in '
                        'mega-cap technology companies, with Apple, Microsoft, and '
                        'Nvidia comprising over 24% of the portfolio. The fund '
                        'tracks the Nasdaq-100, giving it exposure to non-tech '
                        'Nasdaq names like Costco and PepsiCo alongside its '
                        'dominant tech weighting.'
}
```

### AI Disclosure Reminder
Per the course AI policy, your submission must include:
1. Which LLM you used (e.g., Gemini 2.5 Flash-Lite, Ollama Gemma 3 4B)
2. Your prompt template (included in your notebook)
3. A sample of 3–5 generated summaries with your assessment of their accuracy
4. Any hallucinations you caught and how you handled them

---

## Part 5: Data Quality Validation

Before proceeding to similarity calculations, validate your data:

```python
# Check data completeness
print(f"Total ETFs in master: {etf_master_col.count_documents({})}")
print(f"Equity ETFs with holdings: {etf_holdings_col.count_documents({})}")

# Validate holdings data
for etf_doc in etf_holdings_col.find().limit(5):
    total_weight = sum(h['weight'] for h in etf_doc['holdings'])
    has_summary = etf_doc.get('strategy_summary') is not None
    print(f"{etf_doc['etf_ticker']}: {len(etf_doc['holdings'])} holdings, "
          f"total weight: {total_weight:.2f}%, summary: {'✓' if has_summary else '✗'}")
```

Common issues to check:
- Are weights numeric (not strings)? This is critical for MongoDB queries!
- Do weights sum to ~100%?
- Are stock tickers properly cleaned (no spaces, consistent format)?
- Zero Jaccard similarity between similar ETFs (e.g., QQQ and XLK) indicates ticker format issues
- Do strategy summaries exist for all ETFs? If not, run the self-healing pipeline again.

---

## Part 6: Recommending Similar ETFs

### Similarity Metrics
Implement **all three** of the following approaches and compare the results:

**Metric 1: Jaccard Similarity** (binary — does the ETF hold this stock or not?)
```python
def jaccard(set1, set2):
    return len(set1.intersection(set2)) / len(set1.union(set2))
```

**Metric 2: Cosine Similarity on Holdings Weights** (numeric — how much of each stock?)
```python
from scipy.spatial.distance import cosine
similarity = 1.0 - cosine(weights1, weights2)
```

**Metric 3: Cosine Similarity on Strategy Summary Embeddings** (semantic — what does the fund *do*?)

This is where Part 4 pays off. Use a sentence transformer to encode each ETF's `strategy_summary` into a vector, then compute cosine similarity on those embeddings. This captures semantic similarity that holdings-based metrics miss — for example, two ETFs might hold completely different stocks but both pursue a "large-cap value" strategy.

```python
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Encode all strategy summaries
model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)

# Pull summaries from MongoDB
etf_docs = list(etf_holdings_col.find({'query_date': today_str, 'strategy_summary': {'$ne': None}}))
tickers = [doc['etf_ticker'] for doc in etf_docs]
summaries = [doc['strategy_summary'] for doc in etf_docs]

embeddings = model.encode(summaries)

# Find most similar to a given ETF
query_idx = tickers.index('QQQ')
scores = cosine_similarity([embeddings[query_idx]], embeddings)[0]
```

It is optional but encouraged to store the embeddings back in MongoDB (as a list of floats in each document). If you prefer to keep the embeddings in a pandas DataFrame or numpy array, that is also acceptable.

### Expected Results
For validation, check these known similar pairs:
- QQQ and XLK: Jaccard ~0.2, Holdings Cosine ~0.83
- SPY and VOO: Should be nearly identical across all three metrics
- If you get 0 similarity for obviously related ETFs, check stock ticker cleaning!

### Compare the Three Metrics
In your presentation, discuss:
- Where do the three metrics **agree**? (e.g., SPY and VOO should be top matches everywhere)
- Where do they **disagree**? (e.g., two sector ETFs with different holdings but similar strategies)
- Which metric would you recommend to the Head of Product and why?

### Implementation
Create a function that:
1. Takes an ETF ticker as input
2. Calculates all three similarity metrics against other ETFs
3. Returns top N most similar ETFs
4. Displays the strategy summaries side-by-side so an advisor can explain the comparison

---

## Parts 7–10: Optional Extensions (Extra Credit)

### Part 7: Expense Ratio Analysis with Selenium
When comparing similar ETFs, expense ratio is crucial for investors. Use Selenium to scrape expense ratios from https://stockanalysis.com/etf/screener/ and enhance your recommender to highlight fee savings.

### Part 8: Investment Ideas with Apriori Algorithm
Cathy Wood has hired your firm to provide investment ideas for her ARKK innovation ETF. Use market basket analysis to discover stocks that frequently co-occur with ARKK holdings in other ETFs. See the `mlxtend` library for implementation.

See: https://colab.research.google.com/drive/1e3ZC_c-K5ikklU2CP2NRdfd3L7-NcoHy#scrollTo=U4pNfFp9A7fC

### Part 9: Stock Factor Exposures
Calculate factor exposures (size, value, momentum, quality, volatility) for each stock using Finviz data, then aggregate to the ETF level using holdings weights. See the project template for detailed factor mapping.

### Part 10: ETF Clustering by Factor Exposures
Cluster ETFs based on their factor exposures using K-Means and visualize with PCA to identify style groups (e.g., small-cap value vs large-cap growth).

Part 9 and 10 References
* Example of Morningstar Factors and Clustering: https://indexes.morningstar.com/indexes/details/morningstar-us-large-growth-FSUSA00KGU?currency=USD&variant=TR&tab=portfolio
* See: https://colab.research.google.com/drive/1dyRox4XCL_5ly2O-zItByFaCSpkuH_2E?usp=sharing

---

## Grading Guidelines

### Minimum Requirements for Full Credit

**Part 1 (15%): ETF Master Collection**
- Successfully scrape at least 2000 ETFs from Finviz (or equivalent)
- Proper data cleaning and type conversion
- Include query_date in all documents
- Exception handling that prevents crashes

**Part 2 (25%): Holdings Collection**
- Obtain holdings for at least 100 Equity ETFs
- Use multiple data sources (at least 3: iShares, Invesco, SSGA)
- Implement resume capability (don't re-fetch existing data)
- Proper distinction between etf_ticker and stock_ticker
- Using the historical SSGA snapshots, show how at least one ETF's holdings changed over time

**Part 3 (15%): MongoDB Implementation**
- Proper indexes including compound index on (query_date, etf_ticker)
- Demonstrate query performance improvement with indexes
- Use bulk operations where appropriate
- No hardcoded credentials

**Part 4 (15%): AI-Generated Strategy Summaries**
- Working LLM integration (Gemini API or Ollama)
- Self-healing pipeline that resumes after rate limits
- Summaries stored in MongoDB alongside holdings
- AI disclosure statement with accuracy assessment of generated summaries
- Evidence of catching/handling at least one hallucination

**Part 5 (10%): Data Quality & Cleaning**
- Handle missing/malformed data gracefully
- Ensure numeric types for weights
- Consistent data formats across sources
- Clean stock ticker symbols

**Part 6 (10%): Similarity Recommendations**
- Implement all 3 similarity metrics: Jaccard, cosine on holdings weights, cosine on strategy summary embeddings
- Use a sentence transformer (e.g., `nomic-embed-text-v1.5`) to encode the Part 4 summaries
- Return meaningful recommendations with strategy summaries displayed side-by-side
- Compare where the three metrics agree and disagree
- Validate with known similar ETF pairs

**Presentation (10%)**
- 5–10 minute video highlighting key findings
- Each group member speaks for equal time
- Demo of the recommender system with example queries
- Be prepared for an oral examination on any part of the project

### Bonus Points
- Expense ratio scraping and fee-savings analysis in recommendations
- Apriori association rules for ARKK
- Factor exposures and ETF clustering
- Creative visualization of recommendations or holdings drift over time
- Performance optimizations (caching, parallel processing)

### Common Deductions
- Hardcoded credentials or API keys (-5%)
- No exception handling causing crashes (-10%)
- Poor MongoDB document design making queries inefficient (-10%)
- Not implementing resume capability (-5%)
- Data quality issues (spaces in tickers, string weights) (-5%)
- Missing query_date in documents (-5%)
- Confusing ETF tickers with stock tickers (-5%)
- Missing AI disclosure statement (-5%)
- LLM summaries not validated for accuracy (-5%)

---

## Submission Requirements
1. Google Colab notebook (viewable link) with all implementations and names of each group member
2. Video presentation (5–10 minutes) — file or link
3. MongoDB screenshots showing document counts, sample documents (including strategy_summary), and indexes
4. Trello board with tasks assigned and completed
5. AI disclosure statement documenting LLM usage
6. Example recommendations for at least 3 test ETFs (e.g., SPY, QQQ, ARKK) with side-by-side strategy summaries


### **Appendix: Prompt Engineering Best Practices (Reference)**

How you ask the LLM *matters*. A well-crafted prompt is the difference between getting a perfect JSON output and a useless, conversational paragraph. Here are the core principles you should be using.

1.  **Role Specification (Persona):** Tell the AI *what it is*. This sets the context and tone.

      * **Bad:** "Summarize this."
      * **Good:** "You are a highly skilled data analyst and finance professional."

2.  **Task Description:** Be explicit about the *goal*, not just the action.

      * **Bad:** "Find the key metrics."
      * **Good:** "Your job is to extract key financial metrics... The goal is to populate our database with the correct quarterly revenue, net income, and EPS."

3.  **Use Delimiters:** Clearly separate your instructions from the data you want processed. Use markers like `---`, `"""`, or `###`.

    ```
    Summarize the text provided inside the triple dashes.
    ---
    {your text here}
    ---
    ```

4.  **Specify Constraints:** Tell the model exactly what you *want* and what you *don't want*.

      * "Provide a summary in three bullet points."
      * "Do not use any markdown formatting."
      * "The response must not include a preamble like 'Here is the summary...'"

5.  **Provide Examples (Few-Shot):** *Show*, don't just tell. Providing 2-3 examples of the input and desired output is the single most effective way to get a reliable format.

      * **Example 1:** Input: '...credit is AAA.' -\> Output: `{"rating": "AAA"}`
      * **Example 2:** Input: '...rating of P-1.' -\> Output: `{"rating": "P-1"}`

6.  **Define the Output Format:** Be explicit. For this project, you **must** request a specific JSON structure. This is what Pydantic helps you enforce.

      * **Good:** "The output must be structured as a JSON object with the keys `company_name` and `material_points`."

7.  **Instruct on Error Handling:** Tell the AI what to do when it fails or data is missing. This prevents it from making up ("hallucinating") an answer.

      * **Good:** "If the document does not contain any material information, return an empty list for the `material_points` key."